# HPI-GCN-RP: Fine-tuning with Custom Fight Data (Kaggle)

**Fine-tune** the pre-trained HPI-GCN-RP model using your custom fight/push videos.

This notebook does **everything on Kaggle** — extraction + fine-tuning + export.

---

### How to use:

**Step 1 — Upload 3 datasets to Kaggle:**
1. **Dataset 1**: Original NTU skeleton `.npy` files (with `train/fight/` and `train/nonfight/`)
2. **Dataset 2**: Your 32 fight videos (`.mp4` files)
3. **Dataset 3**: `best_model.pth` (pre-trained model weights)

**Step 2 — Create notebook on Kaggle:**
1. Click **+ New Notebook**
2. **+ Add Input** → add all 3 datasets
3. **File → Import Notebook** → upload this `.ipynb`

**Step 3 — Enable GPU + Internet:**
- Settings → Accelerator → **GPU P100** or **T4 x2**
- Settings → **Internet** → **On** (for ultralytics install)

**Step 4 — Run All** (Ctrl+F9)
- Skeleton extraction from videos (~5-10 min)
- Fine-tuning (~20-40 min)

**Step 5 — Download** `hpi_gcn_rp_finetuned.pt` from Output tab
- Rename to `best_model.pth`
- Replace `AiGuardian/models/best_model.pth`

## 0. Configuration & Imports

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import deque
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'Memory: {mem / 1e9:.1f} GB')

In [ ]:
# ── Data paths (auto-detected on Kaggle) ──
ORIGINAL_SKELETON_DIR = None
CUSTOM_SKELETON_DIR = None
PRETRAINED_MODEL_PATH = None

# ── Model config (must match pre-trained) ──
SEQUENCE_LEN = 30
NUM_KEYPOINTS = 17
NUM_PEOPLE = 2
IN_CHANNELS = 3
NUM_NODES = NUM_KEYPOINTS * NUM_PEOPLE  # 34
FEATURE_DIM = NUM_NODES * IN_CHANNELS   # 102

# ── Fine-tuning hyperparameters ──
BATCH_SIZE = 16
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 25
WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.1
DROPOUT = 0.5
CUSTOM_OVERSAMPLE = 3   # was 5 — reduced to prevent overfitting on custom data
REPEAT_DATASET = 3       # was 10 — 3×3=9x repeat vs old 5×10=50x

# Phase 1 LR
LR_HEAD_P1 = 2e-4
LR_GCN_LATE_P1 = 1e-4

# Phase 2 LR (discriminative)
LR_GCN_EARLY = 1e-5
LR_GCN_MID = 5e-5
LR_GCN_LATE = 1e-4
LR_HEAD = 2e-4

# Focal loss
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0

# Early stopping
PATIENCE = 8

# MixUp augmentation
MIXUP_ALPHA = 0.3

# NTU cross-subject split
TRAIN_SUBJECTS = {1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38}

print("Config loaded.")

## 0.5 Extract Skeletons from Fight Videos (YOLO)

This section extracts skeleton data from your raw `.mp4` fight videos using YOLO-pose.
- Finds all `.mp4` files in `/kaggle/input/`
- Extracts keypoints with sliding window (multiple clips per video)
- Saves `.npy` files to `/kaggle/working/skeletons/custom/train/fight/`

In [ ]:
!pip install ultralytics -q
print("ultralytics installed.")

In [ ]:
import cv2
from ultralytics import YOLO

# ── Extraction config ──
YOLO_CONF = 0.3
MAX_PEOPLE = 2
SLIDING_STRIDE = 8  # frames between windows

# ── Find all .mp4 fight videos in /kaggle/input/ ──
video_files = []
if os.path.exists("/kaggle/input"):
    for f in Path("/kaggle/input").rglob("*"):
        if f.suffix.lower() in ('.mp4', '.avi', '.mkv', '.mov'):
            video_files.append(f)

print(f"Found {len(video_files)} video files in /kaggle/input/")
for v in video_files[:5]:
    print(f"  {v}")
if len(video_files) > 5:
    print(f"  ... and {len(video_files) - 5} more")

if len(video_files) == 0:
    print("\nWARNING: No video files found! Skipping extraction.")
    print("If you uploaded pre-extracted .npy files, this is fine — they will be detected below.")

In [ ]:
def extract_frame_features(yolo_model, frame, yolo_device, use_half):
    """Extract (102,) skeleton feature vector from a single frame."""
    h, w = frame.shape[:2]
    results = yolo_model(frame, verbose=False, conf=YOLO_CONF, imgsz=640, half=use_half)

    frame_kps = []
    if results[0].keypoints is not None and len(results[0].keypoints) > 0:
        kp_data = results[0].keypoints.data.cpu().numpy()
        for person_kp in kp_data:
            if len(person_kp) == 0:
                continue
            normalized = person_kp.copy()
            normalized[:, 0] /= w
            normalized[:, 1] /= h
            frame_kps.append(normalized.flatten())

    while len(frame_kps) < MAX_PEOPLE:
        frame_kps.append(np.zeros(NUM_KEYPOINTS * IN_CHANNELS, dtype=np.float32))
    frame_kps = frame_kps[:MAX_PEOPLE]

    # Sort left-to-right
    if len(frame_kps) >= 2:
        kp0 = frame_kps[0].reshape(NUM_KEYPOINTS, IN_CHANNELS)
        kp1 = frame_kps[1].reshape(NUM_KEYPOINTS, IN_CHANNELS)
        v0 = kp0[kp0[:, 2] > 0.3]
        v1 = kp1[kp1[:, 2] > 0.3]
        c0 = np.mean(v0[:, 0]) if len(v0) > 0 else 0.5
        c1 = np.mean(v1[:, 0]) if len(v1) > 0 else 0.5
        if c0 > c1:
            frame_kps[0], frame_kps[1] = frame_kps[1], frame_kps[0]

    return np.concatenate(frame_kps)


def extract_sliding_window(yolo_model, video_path, yolo_device, stride=8):
    """Extract overlapping (30, 102) clips from a video using sliding window."""
    use_half = yolo_device != "cpu"
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < 5:
        cap.release()
        return []

    # Read ALL frames
    all_features = []
    for _ in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        all_features.append(extract_frame_features(yolo_model, frame, yolo_device, use_half))
    cap.release()

    n = len(all_features)
    if n == 0:
        return []

    # Loop-pad if shorter than SEQUENCE_LEN
    if n < SEQUENCE_LEN:
        padded = [all_features[i % n] for i in range(SEQUENCE_LEN)]
        return [np.array(padded, dtype=np.float32)]

    # Sliding window
    clips = []
    for start in range(0, n - SEQUENCE_LEN + 1, stride):
        clip = all_features[start:start + SEQUENCE_LEN]
        clips.append(np.array(clip, dtype=np.float32))

    # Include last window if not covered
    if (n - SEQUENCE_LEN) % stride != 0:
        clip = all_features[n - SEQUENCE_LEN:]
        clips.append(np.array(clip, dtype=np.float32))

    return clips


# ── Run extraction ──
CUSTOM_OUTPUT = "/kaggle/working/skeletons/custom/train/fight"

if len(video_files) > 0:
    os.makedirs(CUSTOM_OUTPUT, exist_ok=True)

    # Load YOLO pose model
    yolo_device = "cuda" if torch.cuda.is_available() else "cpu"
    yolo_model = YOLO("yolo11n-pose.pt")
    if yolo_device == "cuda":
        yolo_model.to("cuda")
    print(f"YOLO device: {yolo_device}")

    total_clips = 0
    for i, vf in enumerate(video_files):
        # Check if already extracted
        existing = list(Path(CUSTOM_OUTPUT).glob(f"{vf.stem}_win*.npy"))
        if existing:
            total_clips += len(existing)
            continue

        clips = extract_sliding_window(yolo_model, vf, yolo_device, stride=SLIDING_STRIDE)
        for ci, clip in enumerate(clips):
            np.save(os.path.join(CUSTOM_OUTPUT, f"{vf.stem}_win{ci}.npy"), clip)
        total_clips += len(clips)

        if (i + 1) % 5 == 0 or i == len(video_files) - 1:
            print(f"  [{i+1}/{len(video_files)}] {vf.stem}: {len(clips)} clips (total: {total_clips})")

    # Clean up YOLO to free GPU memory
    del yolo_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\nExtraction done: {total_clips} clips from {len(video_files)} videos")
    print(f"Output: {CUSTOM_OUTPUT}")

    # Verify
    npy_count = len(list(Path(CUSTOM_OUTPUT).glob("*.npy")))
    if npy_count > 0:
        sample = np.load(str(list(Path(CUSTOM_OUTPUT).glob("*.npy"))[0]))
        print(f"Verified: {npy_count} .npy files, shape={sample.shape}")
else:
    print("No videos to extract, skipping.")

## 0.6 Process NTU Skeleton Files (if raw .skeleton files found)

If you uploaded the raw NTU RGB+D dataset (`.skeleton` files), this section converts them to `.npy` skeleton format.
This takes ~10-20 min for the full NTU dataset.

In [ ]:
# ── NTU RGB+D .skeleton → .npy conversion ──

NTU_TO_COCO = {0:3, 1:3, 2:3, 3:3, 4:3, 5:4, 6:8, 7:5, 8:9, 9:6, 10:10, 11:12, 12:16, 13:13, 14:17, 15:14, 16:18}
# Expanded with NTU 120 classes
NTU_FIGHT_CLASSES = {50, 51, 52, 106, 108}
NTU_NORMAL_CLASSES = {53, 54, 55, 56, 57, 58, 59, 60, 63, 99, 112}
NTU_SELECTED = NTU_FIGHT_CLASSES | NTU_NORMAL_CLASSES


def parse_ntu_skeleton(filepath):
    """Parse NTU RGB+D .skeleton file -> list of frames, each frame = list of bodies (25,3)."""
    frames = []
    try:
        with open(filepath, 'r') as f:
            num_frames = int(f.readline().strip())
            for _ in range(num_frames):
                num_bodies = int(f.readline().strip())
                bodies = []
                for _ in range(num_bodies):
                    _ = f.readline()  # body info line
                    num_joints = int(f.readline().strip())
                    joints = np.zeros((num_joints, 3), dtype=np.float32)
                    for j in range(num_joints):
                        parts = f.readline().strip().split()
                        joints[j] = [float(parts[0]), float(parts[1]), float(parts[2])]
                    bodies.append(joints)
                frames.append(bodies)
    except Exception:
        return []
    return frames


def ntu_to_coco_skeleton(ntu_joints_25):
    """Convert NTU 25-joint (25,3) -> COCO 17-joint (17,3) normalized."""
    coco = np.zeros((17, 3), dtype=np.float32)
    for coco_idx, ntu_idx in NTU_TO_COCO.items():
        if ntu_idx < len(ntu_joints_25):
            pt = ntu_joints_25[ntu_idx]
            coco[coco_idx, 0] = (pt[0] + 1.5) / 3.0  # normalize x to ~[0,1]
            coco[coco_idx, 1] = (pt[1] + 0.5) / 2.0  # normalize y to ~[0,1]
            coco[coco_idx, 2] = 1.0  # confidence = 1 for ground truth
    return coco


def get_ntu_action_class(filename):
    """Extract action class number from NTU filename like S001C001P001R001A050."""
    name = Path(filename).stem
    a_idx = name.find('A')
    if a_idx >= 0:
        try:
            return int(name[a_idx+1:a_idx+4])
        except ValueError:
            pass
    return -1


def get_ntu_subject(filename):
    """Extract subject number from NTU filename."""
    name = Path(filename).stem
    p_idx = name.find('P')
    if p_idx >= 0:
        try:
            return int(name[p_idx+1:p_idx+4])
        except ValueError:
            pass
    return -1


# ── Find .skeleton files ──
skeleton_files = []
if os.path.exists("/kaggle/input"):
    for f in Path("/kaggle/input").rglob("*.skeleton"):
        action = get_ntu_action_class(f.name)
        if action in NTU_SELECTED:
            skeleton_files.append(f)

print(f"Found {len(skeleton_files)} NTU .skeleton files (fight+nonfight actions)")
print(f"  Fight classes: {sorted(NTU_FIGHT_CLASSES)} (A050 punch, A051 kick, A052 push, A106 hit object, A108 knock over)")
print(f"  Normal classes: {sorted(NTU_NORMAL_CLASSES)} (A053-A060, A063, A099, A112)")

# ── Process if found ──
NTU_OUTPUT_DIR = "/kaggle/working/skeletons/ntu"

if len(skeleton_files) > 0:
    fight_dir = Path(NTU_OUTPUT_DIR) / "train" / "fight"
    nonfight_dir = Path(NTU_OUTPUT_DIR) / "train" / "nonfight"
    fight_dir.mkdir(parents=True, exist_ok=True)
    nonfight_dir.mkdir(parents=True, exist_ok=True)

    stats = {"fight": 0, "nonfight": 0, "failed": 0}

    for i, sf in enumerate(skeleton_files):
        action = get_ntu_action_class(sf.name)

        if action in NTU_FIGHT_CLASSES:
            out_dir = fight_dir
            label = "fight"
        else:
            out_dir = nonfight_dir
            label = "nonfight"

        out_file = out_dir / f"{sf.stem}.npy"
        if out_file.exists():
            stats[label] += 1
            continue

        frames = parse_ntu_skeleton(str(sf))
        if len(frames) < 5:
            stats["failed"] += 1
            continue

        # Sample 30 frames evenly
        indices = np.linspace(0, len(frames) - 1, SEQUENCE_LEN, dtype=int)
        all_features = []

        for idx in indices:
            bodies = frames[idx]
            frame_kps = []

            for body in bodies[:NUM_PEOPLE]:
                if len(body) >= 25:
                    coco_kp = ntu_to_coco_skeleton(body)
                    frame_kps.append(coco_kp.flatten())

            while len(frame_kps) < NUM_PEOPLE:
                frame_kps.append(np.zeros(NUM_KEYPOINTS * IN_CHANNELS, dtype=np.float32))
            frame_kps = frame_kps[:NUM_PEOPLE]

            # Sort people left-to-right by X centroid (must match video extraction & inference)
            if len(frame_kps) >= 2:
                kp0 = frame_kps[0].reshape(NUM_KEYPOINTS, IN_CHANNELS)
                kp1 = frame_kps[1].reshape(NUM_KEYPOINTS, IN_CHANNELS)
                vis0 = kp0[kp0[:, 2] > 0.3]
                vis1 = kp1[kp1[:, 2] > 0.3]
                cx0 = np.mean(vis0[:, 0]) if len(vis0) > 0 else 0.5
                cx1 = np.mean(vis1[:, 0]) if len(vis1) > 0 else 0.5
                if cx0 > cx1:
                    frame_kps[0], frame_kps[1] = frame_kps[1], frame_kps[0]

            all_features.append(np.concatenate(frame_kps))

        result = np.array(all_features, dtype=np.float32)
        np.save(str(out_file), result)
        stats[label] += 1

        if (i + 1) % 500 == 0 or i == len(skeleton_files) - 1:
            print(f"  [{i+1}/{len(skeleton_files)}] fight={stats['fight']}, nonfight={stats['nonfight']}, failed={stats['failed']}")

    print(f"\nNTU processing done:")
    print(f"  Fight: {stats['fight']}")
    print(f"  Non-fight: {stats['nonfight']}")
    print(f"  Failed: {stats['failed']}")
    print(f"  Output: {NTU_OUTPUT_DIR}")
else:
    print("\nNo NTU .skeleton files found in /kaggle/input/.")
    print("If you uploaded pre-processed .npy files, they will be detected in the next cell.")
    print("If not, the model needs BOTH fight and nonfight data to train properly.")

In [ ]:
# Auto-detect datasets on Kaggle
# Search /kaggle/input/ (uploaded), /kaggle/working/skeletons/ (extracted video + NTU)
search_dirs = []
if os.path.exists("/kaggle/input"):
    search_dirs.append(Path("/kaggle/input"))
if os.path.exists("/kaggle/working/skeletons"):
    search_dirs.append(Path("/kaggle/working/skeletons"))

for search_root in search_dirs:
    for dataset_dir in sorted(search_root.rglob("*")):
        if not dataset_dir.is_dir():
            continue
        # Only check leaf directories that contain .npy files directly
        npy_files = list(dataset_dir.glob("*.npy"))
        if not npy_files:
            continue
        # Walk up to find root with train/val structure
        parent_path = str(dataset_dir)
        has_nonfight = "nonfight" in parent_path.lower()
        has_fight = "fight" in parent_path.lower() and "nonfight" not in parent_path.lower()

        if has_nonfight:
            # This is a nonfight directory — find the root (parent of train/)
            if ORIGINAL_SKELETON_DIR is None:
                parts = dataset_dir.parts
                for j, p in enumerate(parts):
                    if p in ("train", "val"):
                        ORIGINAL_SKELETON_DIR = str(Path(*parts[:j]))
                        break
                if ORIGINAL_SKELETON_DIR is None:
                    ORIGINAL_SKELETON_DIR = str(dataset_dir.parent.parent)
                print(f"Found original skeletons: {ORIGINAL_SKELETON_DIR}")
        elif has_fight:
            if CUSTOM_SKELETON_DIR is None:
                parts = dataset_dir.parts
                for j, p in enumerate(parts):
                    if p in ("train", "val"):
                        CUSTOM_SKELETON_DIR = str(Path(*parts[:j]))
                        break
                if CUSTOM_SKELETON_DIR is None:
                    CUSTOM_SKELETON_DIR = str(dataset_dir.parent.parent)
                print(f"Found custom fight skeletons: {CUSTOM_SKELETON_DIR}")

# Direct checks for known output paths
if ORIGINAL_SKELETON_DIR is None and os.path.exists("/kaggle/working/skeletons/ntu"):
    npy_check = list(Path("/kaggle/working/skeletons/ntu").rglob("*.npy"))
    if npy_check:
        ORIGINAL_SKELETON_DIR = "/kaggle/working/skeletons/ntu"
        print(f"Found NTU processed skeletons: {ORIGINAL_SKELETON_DIR} ({len(npy_check)} files)")

if CUSTOM_SKELETON_DIR is None and os.path.exists("/kaggle/working/skeletons/custom"):
    npy_check = list(Path("/kaggle/working/skeletons/custom").rglob("*.npy"))
    if npy_check:
        CUSTOM_SKELETON_DIR = "/kaggle/working/skeletons/custom"
        print(f"Found extracted custom skeletons: {CUSTOM_SKELETON_DIR} ({len(npy_check)} files)")

# Find pretrained model
if PRETRAINED_MODEL_PATH is None and os.path.exists("/kaggle/input"):
    for ext in ("*.pth", "*.pt"):
        for pth in Path("/kaggle/input").rglob(ext):
            if "yolo" not in pth.name.lower():
                PRETRAINED_MODEL_PATH = str(pth)
                print(f"Found pretrained model: {PRETRAINED_MODEL_PATH}")
                break
        if PRETRAINED_MODEL_PATH:
            break

# Fallbacks
if ORIGINAL_SKELETON_DIR is None:
    ORIGINAL_SKELETON_DIR = "./skeletons/ntu"
    print(f"WARNING: No original skeletons found! Using local path: {ORIGINAL_SKELETON_DIR}")
if CUSTOM_SKELETON_DIR is None:
    CUSTOM_SKELETON_DIR = "./skeletons/custom"
    print(f"WARNING: No custom skeletons found! Using local path: {CUSTOM_SKELETON_DIR}")
if PRETRAINED_MODEL_PATH is None:
    PRETRAINED_MODEL_PATH = "./models/best_model.pth"
    print(f"WARNING: No pretrained model found! Using local path: {PRETRAINED_MODEL_PATH}")

print(f"\nOriginal data:     {ORIGINAL_SKELETON_DIR}")
print(f"Custom fight data: {CUSTOM_SKELETON_DIR}")
print(f"Pretrained model:  {PRETRAINED_MODEL_PATH}")

## 1. Graph Construction

In [ ]:
def build_coco_adjacency(num_keypoints=17, num_people=2):
    """Build 34x34 adjacency matrix for 2-person COCO skeletons."""
    V = num_keypoints * num_people

    intra_edges = [
        (0, 1), (0, 2), (1, 3), (2, 4),
        (5, 6),
        (5, 7), (7, 9),
        (6, 8), (8, 10),
        (5, 11), (6, 12),
        (11, 12),
        (11, 13), (13, 15),
        (12, 14), (14, 16),
    ]

    edges = []
    for i, j in intra_edges:
        edges.append((i, j))
    for i, j in intra_edges:
        edges.append((i + 17, j + 17))
    for i in range(17):
        edges.append((i, i + 17))

    adj = {i: set() for i in range(V)}
    for i, j in edges:
        adj[i].add(j)
        adj[j].add(i)

    dist = [float('inf')] * V
    queue = deque()
    for c in [0, 17]:
        dist[c] = 0
        queue.append(c)
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if dist[v] > dist[u] + 1:
                dist[v] = dist[u] + 1
                queue.append(v)

    A_self = np.eye(V, dtype=np.float32)
    A_inward = np.zeros((V, V), dtype=np.float32)
    A_outward = np.zeros((V, V), dtype=np.float32)

    for i, j in edges:
        if dist[i] < dist[j]:
            A_inward[j][i] = 1
            A_outward[i][j] = 1
        elif dist[j] < dist[i]:
            A_inward[i][j] = 1
            A_outward[j][i] = 1
        else:
            A_inward[i][j] = 1
            A_inward[j][i] = 1

    def _normalize(A_part):
        D = np.sum(A_part, axis=1)
        D_inv_sqrt = np.diag(np.where(D > 0, 1.0 / np.sqrt(D), 0))
        return D_inv_sqrt @ A_part @ D_inv_sqrt

    A = np.stack([_normalize(A_self), _normalize(A_inward), _normalize(A_outward)])
    print(f"Adjacency matrix: {A.shape}")
    return A

A_partitions = build_coco_adjacency()

## 2. Model Architecture

In [ ]:
class SpatialGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, A, num_partitions=3):
        super().__init__()
        self.num_partitions = num_partitions
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        self.conv = nn.Conv2d(in_channels, out_channels * num_partitions, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        B, C, T, V = x.shape
        x_conv = self.conv(x)
        C_out = x_conv.shape[1] // self.num_partitions
        x_conv = x_conv.reshape(B, self.num_partitions, C_out, T, V)
        out = torch.zeros(B, C_out, T, V, device=x.device)
        for k in range(self.num_partitions):
            out = out + torch.einsum('bctv,vw->bctw', x_conv[:, k], self.A[k])
        return self.bn(out)


class RelationPrediction(nn.Module):
    def __init__(self, in_channels, num_nodes=34):
        super().__init__()
        self.B = nn.Parameter(torch.zeros(num_nodes, num_nodes))
        self.W_q = nn.Conv2d(in_channels, max(in_channels // 4, 1), kernel_size=1)
        self.W_k = nn.Conv2d(in_channels, max(in_channels // 4, 1), kernel_size=1)

    def forward(self, x):
        B, C, T, V = x.shape
        x_pool = x.mean(dim=2, keepdim=True)
        q = self.W_q(x_pool).squeeze(2)
        k = self.W_k(x_pool).squeeze(2)
        d = q.shape[1]
        attention = torch.bmm(q.permute(0, 2, 1), k) / (d ** 0.5)
        attention = F.softmax(attention, dim=-1)
        C_adaptive = attention + self.B.unsqueeze(0)
        return torch.einsum('bctv,bvw->bctw', x, C_adaptive)


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.1):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_ch, out_ch, A)
        self.rp = RelationPrediction(in_ch, num_nodes=A.shape[1])
        self.rp_proj = nn.Conv2d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()
        self.relu1 = nn.ReLU(inplace=True)
        self.tcn = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=(9, 1), stride=(stride, 1), padding=(4, 0)),
            nn.BatchNorm2d(out_ch),
        )
        self.relu2 = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)
        if in_ch != out_ch or stride != 1:
            self.residual = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        out = self.sgcn(x) + self.rp_proj(self.rp(x))
        out = self.relu1(out)
        out = self.tcn(out)
        out = self.relu2(out + res)
        return self.drop(out)


class HPI_GCN_RP(nn.Module):
    MODEL_NAME = "HPI-GCN-RP"

    def __init__(self, num_keypoints=17, num_people=2, in_channels=3,
                 num_classes=1, dropout=0.3):
        super().__init__()
        self.num_nodes = num_keypoints * num_people
        self.num_keypoints = num_keypoints
        self.num_people = num_people
        self.in_channels = in_channels

        A = build_coco_adjacency(num_keypoints, num_people)

        self.bn_input = nn.BatchNorm1d(in_channels * self.num_nodes)

        self.layers = nn.ModuleList([
            STGCNBlock(3,   64,  A, stride=1, dropout=0.1),
            STGCNBlock(64,  64,  A, stride=1, dropout=0.1),
            STGCNBlock(64,  128, A, stride=2, dropout=0.1),
            STGCNBlock(128, 128, A, stride=1, dropout=0.1),
            STGCNBlock(128, 256, A, stride=2, dropout=0.1),
            STGCNBlock(256, 256, A, stride=1, dropout=0.1),
        ])

        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        B = x.size(0)
        T = x.size(1)
        x = x.reshape(B, T, self.num_people, self.num_keypoints, self.in_channels)
        x = x.reshape(B, T, self.num_nodes, self.in_channels)
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(B, self.in_channels * self.num_nodes, T)
        x = self.bn_input(x)
        x = x.reshape(B, self.in_channels, T, self.num_nodes)
        for layer in self.layers:
            x = layer(x)
        x = x.mean(dim=[2, 3])
        x = self.drop(x)
        x = self.fc(x)
        return x.squeeze(-1)

print("Model architecture loaded.")

## 3. Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss — focuses on hard examples, handles class imbalance."""
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean()

print(f"Focal Loss: alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}")

## 4. Dataset with Enhanced Augmentation

In [ ]:
# COCO bone connections for bone-length perturbation
BONE_PAIRS = [
    (5, 7), (7, 9),    # left arm
    (6, 8), (8, 10),   # right arm
    (11, 13), (13, 15), # left leg
    (12, 14), (14, 16), # right leg
    (5, 11), (6, 12),   # torso sides
]
LR_SWAP = [(1,2), (3,4), (5,6), (7,8), (9,10), (11,12), (13,14), (15,16)]


class SkeletonDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment
        n_fight = sum(1 for _, l in self.samples if l == 1)
        n_nonfight = sum(1 for _, l in self.samples if l == 0)
        print(f"  Dataset: {len(self.samples)} total ({n_fight} fight, {n_nonfight} nonfight)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        seq = np.load(path).astype(np.float32)
        if self.augment:
            seq = self._augment(seq)
        return torch.FloatTensor(seq), torch.FloatTensor([label])

    def _augment(self, seq):
        # 1. Random temporal shift (0-3 frames)
        shift = np.random.randint(0, 4)
        if shift > 0:
            seq = np.roll(seq, shift, axis=0)

        # 2. Horizontal flip
        if np.random.random() > 0.5:
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            seq_r[:, :, :, 0] = 1.0 - seq_r[:, :, :, 0]
            for l, r in LR_SWAP:
                seq_r[:, :, [l, r], :] = seq_r[:, :, [r, l], :]
            seq_r[:, [0, 1], :, :] = seq_r[:, [1, 0], :, :]
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        # 3. Gaussian noise
        noise = np.random.normal(0, 0.01, seq.shape).astype(np.float32)
        seq = seq + noise

        # 4. Joint dropout (aggressive: 40% chance, 1-4 joints)
        if np.random.random() > 0.6:
            num_drop = np.random.randint(1, 5)
            person = np.random.randint(0, 2)
            joints = np.random.choice(17, num_drop, replace=False)
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            seq_r[:, person, joints, :] = 0
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        # 5. Random temporal speed
        if np.random.random() > 0.7:
            speed = np.random.uniform(0.8, 1.2)
            indices = np.linspace(0, SEQUENCE_LEN - 1, int(SEQUENCE_LEN * speed))
            indices = np.clip(indices, 0, SEQUENCE_LEN - 1).astype(int)
            stretched = seq[indices]
            if len(stretched) != SEQUENCE_LEN:
                new_idx = np.linspace(0, len(stretched) - 1, SEQUENCE_LEN).astype(int)
                seq = stretched[new_idx]

        # 6. Random rotation (~17 degrees)
        if np.random.random() > 0.5:
            angle = np.random.uniform(-0.3, 0.3)  # radians
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            for p in range(2):
                x = seq_r[:, p, :, 0]
                y = seq_r[:, p, :, 1]
                cx = np.mean(x[x > 0]) if np.any(x > 0) else 0.5
                cy = np.mean(y[y > 0]) if np.any(y > 0) else 0.5
                seq_r[:, p, :, 0] = cos_a * (x - cx) - sin_a * (y - cy) + cx
                seq_r[:, p, :, 1] = sin_a * (x - cx) + cos_a * (y - cy) + cy
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        # 7. Random scale (0.85-1.15)
        if np.random.random() > 0.5:
            scale = np.random.uniform(0.85, 1.15)
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            for p in range(2):
                x = seq_r[:, p, :, 0]
                y = seq_r[:, p, :, 1]
                cx = np.mean(x[x > 0]) if np.any(x > 0) else 0.5
                cy = np.mean(y[y > 0]) if np.any(y > 0) else 0.5
                seq_r[:, p, :, 0] = (x - cx) * scale + cx
                seq_r[:, p, :, 1] = (y - cy) * scale + cy
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        # 8. Random translation (dx/dy up to 0.1)
        if np.random.random() > 0.5:
            dx = np.random.uniform(-0.1, 0.1)
            dy = np.random.uniform(-0.1, 0.1)
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            seq_r[:, :, :, 0] += dx
            seq_r[:, :, :, 1] += dy
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        # 9. Temporal crop + resize
        if np.random.random() > 0.6:
            crop_ratio = np.random.uniform(0.5, 1.0)
            crop_len = max(10, int(SEQUENCE_LEN * crop_ratio))
            start = np.random.randint(0, max(1, SEQUENCE_LEN - crop_len))
            cropped = seq[start:start + crop_len]
            indices = np.linspace(0, len(cropped) - 1, SEQUENCE_LEN).astype(int)
            seq = cropped[indices]

        # 10. Bone length perturbation
        if np.random.random() > 0.7:
            seq_r = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            for parent, child in BONE_PAIRS:
                stretch = np.random.uniform(0.9, 1.1)
                for p in range(2):
                    diff_x = seq_r[:, p, child, 0] - seq_r[:, p, parent, 0]
                    diff_y = seq_r[:, p, child, 1] - seq_r[:, p, parent, 1]
                    seq_r[:, p, child, 0] = seq_r[:, p, parent, 0] + diff_x * stretch
                    seq_r[:, p, child, 1] = seq_r[:, p, parent, 1] + diff_y * stretch
            seq = seq_r.reshape(SEQUENCE_LEN, -1)

        return seq


class RepeatDataset(Dataset):
    """Repeat dataset N times. With augmentation, each repeat yields different data."""
    def __init__(self, dataset, times=10):
        self.dataset = dataset
        self.times = times

    def __len__(self):
        return len(self.dataset) * self.times

    def __getitem__(self, idx):
        return self.dataset[idx % len(self.dataset)]

print("Dataset classes loaded. Augmentations: 10 techniques.")

## 5. Data Loading (Combined Original + Custom)

In [ ]:
def load_skeleton_samples(root_dir):
    samples = []
    root = Path(root_dir)
    if not root.exists():
        print(f"  WARNING: {root_dir} not found!")
        return samples
    for split_dir in sorted(root.iterdir()):
        if not split_dir.is_dir():
            continue
        # Check if this is train/val level or class level directly
        subdirs = [d for d in split_dir.iterdir() if d.is_dir()]
        if subdirs:
            # This is train/ or val/ containing class dirs
            for class_dir in sorted(subdirs):
                label = 1 if 'fight' in class_dir.name.lower() and 'non' not in class_dir.name.lower() else 0
                for npy_file in sorted(class_dir.glob('*.npy')):
                    samples.append((str(npy_file), label))
        else:
            # This IS a class dir (e.g., root/fight/*.npy)
            label = 1 if 'fight' in split_dir.name.lower() and 'non' not in split_dir.name.lower() else 0
            for npy_file in sorted(split_dir.glob('*.npy')):
                samples.append((str(npy_file), label))
    return samples


def get_ntu_subject_from_path(filename):
    stem = Path(filename).stem
    p_idx = stem.find('P')
    if p_idx >= 0:
        try:
            return int(stem[p_idx+1:p_idx+4])
        except ValueError:
            pass
    return -1


# Load original + custom data
original_samples = load_skeleton_samples(ORIGINAL_SKELETON_DIR)
print(f"Original: {len(original_samples)} samples")
print(f"  Fight: {sum(1 for _,l in original_samples if l==1)}")
print(f"  Non-fight: {sum(1 for _,l in original_samples if l==0)}")

custom_samples = load_skeleton_samples(CUSTOM_SKELETON_DIR)
print(f"\nCustom: {len(custom_samples)} samples")
print(f"  Fight: {sum(1 for _,l in custom_samples if l==1)}")
print(f"  Non-fight: {sum(1 for _,l in custom_samples if l==0)}")

# ── SANITY CHECK: need both classes ──
all_samples = original_samples + custom_samples
n_total_fight = sum(1 for _,l in all_samples if l == 1)
n_total_nonfight = sum(1 for _,l in all_samples if l == 0)

if n_total_nonfight == 0:
    raise ValueError(
        "\n\nERROR: No nonfight samples found!\n"
        "The model needs BOTH fight AND nonfight data to learn.\n"
        "Upload the NTU skeleton dataset (.skeleton files) to Kaggle,\n"
        "or upload pre-processed .npy files with train/fight/ and train/nonfight/ folders.\n"
    )
if n_total_fight == 0:
    raise ValueError(
        "\n\nERROR: No fight samples found!\n"
        "Upload your fight video dataset or pre-processed fight .npy files.\n"
    )

print(f"\nTotal: {n_total_fight} fight + {n_total_nonfight} nonfight = {len(all_samples)}")

# NTU cross-subject split for original
train_samples = []
val_samples = []

for path, label in original_samples:
    subject = get_ntu_subject_from_path(path)
    if subject in TRAIN_SUBJECTS:
        train_samples.append((path, label))
    else:
        val_samples.append((path, label))

# Add oversampled custom to train
train_samples.extend(custom_samples * CUSTOM_OVERSAMPLE)

# If no NTU filenames, random 80/20
if len(val_samples) == 0:
    print("\nNo NTU filenames, using random 80/20 split...")
    combined = original_samples + custom_samples * CUSTOM_OVERSAMPLE
    np.random.seed(42)
    indices = np.random.permutation(len(combined))
    split = int(0.8 * len(indices))
    train_samples = [combined[i] for i in indices[:split]]
    val_samples = [combined[i] for i in indices[split:]]

# Hold out 20% custom for val
if len(custom_samples) > 2:
    np.random.seed(42)
    idx = np.random.permutation(len(custom_samples))
    n_val = max(1, int(0.2 * len(idx)))
    val_samples.extend([custom_samples[i] for i in idx[:n_val]])

print(f"\nFinal split:")
print(f"  Train: {len(train_samples)} ({sum(1 for _,l in train_samples if l==1)} fight, {sum(1 for _,l in train_samples if l==0)} nonfight)")
print(f"  Val:   {len(val_samples)} ({sum(1 for _,l in val_samples if l==1)} fight, {sum(1 for _,l in val_samples if l==0)} nonfight)")

print("\nTraining set:")
train_base = SkeletonDataset(train_samples, augment=True)
train_dataset = RepeatDataset(train_base, times=REPEAT_DATASET)
print(f"  Effective size: {len(train_dataset)} (x{REPEAT_DATASET} repeat)")

print("Validation set:")
val_dataset = SkeletonDataset(val_samples, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches/epoch: {len(train_loader)}, Val batches: {len(val_loader)}")

## 6. Load Pre-trained Model

In [ ]:
model = HPI_GCN_RP(
    num_keypoints=NUM_KEYPOINTS,
    num_people=NUM_PEOPLE,
    in_channels=IN_CHANNELS,
    num_classes=1,
    dropout=DROPOUT,
).to(device)

checkpoint = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
    info = f"epoch={checkpoint.get('epoch','?')}, val_acc={checkpoint.get('val_accuracy', checkpoint.get('val_acc','?'))}"
    print(f"Loaded checkpoint: {info}")
else:
    state_dict = checkpoint

missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"Missing keys: {missing}")
if unexpected:
    print(f"Unexpected keys: {unexpected}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nHPI-GCN-RP: {total_params:,} parameters")

dummy = torch.randn(2, SEQUENCE_LEN, FEATURE_DIM).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"Shape test: {dummy.shape} -> {out.shape}")

## 7. Training Functions

In [ ]:
def smooth_labels(labels, smoothing=0.1):
    return labels * (1 - smoothing) + smoothing * 0.5


def mixup_data(x, y, alpha=0.3):
    """MixUp augmentation: interpolate between random pairs of samples."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def train_epoch(model, loader, criterion, optimizer, use_mixup=True):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for sequences, labels in loader:
        sequences = sequences.to(device)
        labels = labels.squeeze().to(device)

        optimizer.zero_grad()

        if use_mixup and MIXUP_ALPHA > 0:
            mixed_seq, y_a, y_b, lam = mixup_data(sequences, labels, MIXUP_ALPHA)
            logits = model(mixed_seq)
            smooth_a = smooth_labels(y_a, LABEL_SMOOTHING)
            smooth_b = smooth_labels(y_b, LABEL_SMOOTHING)
            loss = lam * criterion(logits, smooth_a) + (1 - lam) * criterion(logits, smooth_b)
            # For accuracy tracking, use original labels with threshold
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (lam * (preds == y_a).float() + (1 - lam) * (preds == y_b).float()).sum().item()
        else:
            smooth = smooth_labels(labels, LABEL_SMOOTHING)
            logits = model(sequences)
            loss = criterion(logits, smooth)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_scores = []  # raw sigmoid scores for threshold optimization
    all_logits = []  # raw logits for temperature scaling
    for sequences, labels in loader:
        sequences = sequences.to(device)
        labels = labels.squeeze().to(device)
        logits = model(sequences)
        loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        scores = torch.sigmoid(logits)
        preds = (scores > 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_scores.extend(scores.cpu().numpy())
        all_logits.extend(logits.cpu().numpy())
    f1 = f1_score(all_labels, all_preds)

    # Compute recall (fight detection rate) — critical metric
    tp = sum(1 for p, l in zip(all_preds, all_labels) if p == 1 and l == 1)
    fn = sum(1 for p, l in zip(all_preds, all_labels) if p == 0 and l == 1)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return total_loss / total, correct / total, f1, all_preds, all_labels, all_scores, all_logits, recall

print("Training functions loaded (with MixUp + recall monitoring).")

## 8. Phase 1: Freeze Early Layers

In [ ]:
print("Phase 1: Freezing layers 0-3, training layers 4-5 + head...")
for i, layer in enumerate(model.layers):
    if i < 4:
        for param in layer.parameters():
            param.requires_grad = False
        print(f"  Frozen: layers[{i}]")

# CRITICAL: Keep all BatchNorm trainable
for name, module in model.named_modules():
    if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        for param in module.parameters():
            param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True
for param in model.drop.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nPhase 1: {trainable:,} / {total:,} trainable ({trainable/total:.0%})")

# Collect param groups
late_gcn_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and ('layers.4' in n or 'layers.5' in n)]
head_params = [p for n, p in model.named_parameters()
               if p.requires_grad and ('fc' in n or 'drop' in n or 'bn_input' in n)]
other_params = [p for n, p in model.named_parameters()
                if p.requires_grad and not any(k in n for k in ['layers.4', 'layers.5', 'fc', 'drop', 'bn_input'])]

param_groups_p1 = []
if late_gcn_params:
    param_groups_p1.append({'params': late_gcn_params, 'lr': LR_GCN_LATE_P1})
if head_params:
    param_groups_p1.append({'params': head_params, 'lr': LR_HEAD_P1})
if other_params:
    param_groups_p1.append({'params': other_params, 'lr': LR_GCN_LATE_P1})

optimizer_p1 = torch.optim.AdamW(param_groups_p1, weight_decay=WEIGHT_DECAY)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_p1, T_0=5, T_mult=2, eta_min=1e-6)
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
print(f"Optimizer: AdamW ({len(param_groups_p1)} groups), Loss: Focal")

In [ ]:
print(f"Phase 1: {EPOCHS_PHASE1} epochs (early layers frozen)...")
print(f"{'Ep':>3} | {'TrainLoss':>9} | {'TrainAcc':>8} | {'ValLoss':>8} | {'ValAcc':>7} | {'ValF1':>6} | {'Recall':>6} | {'LR':>10}")
print("-" * 80)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'val_recall': [], 'lr': [], 'phase': []}
best_val_f1 = 0.0
best_val_acc = 0.0
best_val_recall = 0.0
best_epoch = 0
best_scores = []
best_labels = []
best_logits = []

for epoch in range(1, EPOCHS_PHASE1 + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_p1)
    val_loss, val_acc, val_f1, _, val_labels, val_scores, val_logits, val_recall = eval_epoch(model, val_loader, criterion)
    scheduler_p1.step()

    lr = optimizer_p1.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['val_recall'].append(val_recall)
    history['lr'].append(lr)
    history['phase'].append(1)

    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        best_val_recall = val_recall
        best_epoch = epoch
        best_scores = val_scores
        best_labels = val_labels
        best_logits = val_logits
        marker = ' *'
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'val_acc': val_acc, 'val_f1': val_f1, 'val_recall': val_recall,
            'model_name': 'HPI-GCN-RP', 'num_keypoints': NUM_KEYPOINTS,
            'num_people': NUM_PEOPLE, 'in_channels': IN_CHANNELS,
            'sequence_len': SEQUENCE_LEN, 'fine_tuned': True,
        }, 'best_finetuned.pt')

    print(f"{epoch:>3} | {train_loss:>9.4f} | {train_acc:>7.1%} | {val_loss:>8.4f} | {val_acc:>6.1%} | {val_f1:>5.3f} | {val_recall:>5.1%} | {lr:>10.6f}{marker}")

    # SAFETY: warn if recall drops below 85%
    if val_recall < 0.85:
        print(f"  ⚠️  WARNING: Recall={val_recall:.1%} < 85%! Model may be missing fights.")

print(f"\nPhase 1 done. Best: ep {best_epoch}, f1={best_val_f1:.3f}, acc={best_val_acc:.1%}, recall={best_val_recall:.1%}")

## 9. Phase 2: Unfreeze All, Discriminative LR

In [ ]:
print("Phase 2: Unfreezing all layers...")

for param in model.parameters():
    param.requires_grad = True

param_groups_p2 = [
    {'params': [p for n, p in model.named_parameters() if 'layers.0' in n or 'layers.1' in n], 'lr': LR_GCN_EARLY},
    {'params': [p for n, p in model.named_parameters() if 'layers.2' in n or 'layers.3' in n], 'lr': LR_GCN_MID},
    {'params': [p for n, p in model.named_parameters() if 'layers.4' in n or 'layers.5' in n], 'lr': LR_GCN_LATE},
    {'params': [p for n, p in model.named_parameters() if 'fc' in n or 'drop' in n or 'bn_input' in n], 'lr': LR_HEAD},
]
param_groups_p2 = [g for g in param_groups_p2 if len(list(g['params'])) > 0]

optimizer_p2 = torch.optim.AdamW(param_groups_p2, weight_decay=WEIGHT_DECAY)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_p2, T_0=10, T_mult=2, eta_min=1e-6)

for i, g in enumerate(param_groups_p2):
    n = sum(p.numel() for p in g['params'])
    print(f"  Group {i}: lr={g['lr']}, params={n:,}")

In [ ]:
print(f"\nPhase 2: {EPOCHS_PHASE2} epochs (all layers, discriminative LR)...")
print(f"{'Ep':>3} | {'TrainLoss':>9} | {'TrainAcc':>8} | {'ValLoss':>8} | {'ValAcc':>7} | {'ValF1':>6} | {'Recall':>6} | {'LR':>10}")
print("-" * 80)

no_improve = 0

for epoch in range(1, EPOCHS_PHASE2 + 1):
    global_epoch = EPOCHS_PHASE1 + epoch
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_p2)
    val_loss, val_acc, val_f1, _, val_labels, val_scores, val_logits, val_recall = eval_epoch(model, val_loader, criterion)
    scheduler_p2.step()

    lr = optimizer_p2.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['val_recall'].append(val_recall)
    history['lr'].append(lr)
    history['phase'].append(2)

    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        best_val_recall = val_recall
        best_epoch = global_epoch
        best_scores = val_scores
        best_labels = val_labels
        best_logits = val_logits
        no_improve = 0
        marker = ' *'
        torch.save({
            'epoch': global_epoch, 'model_state_dict': model.state_dict(),
            'val_acc': val_acc, 'val_f1': val_f1, 'val_recall': val_recall,
            'model_name': 'HPI-GCN-RP', 'num_keypoints': NUM_KEYPOINTS,
            'num_people': NUM_PEOPLE, 'in_channels': IN_CHANNELS,
            'sequence_len': SEQUENCE_LEN, 'fine_tuned': True,
        }, 'best_finetuned.pt')
    else:
        no_improve += 1

    print(f"{global_epoch:>3} | {train_loss:>9.4f} | {train_acc:>7.1%} | {val_loss:>8.4f} | {val_acc:>6.1%} | {val_f1:>5.3f} | {val_recall:>5.1%} | {lr:>10.6f}{marker}")

    # SAFETY: warn if recall drops below 85%
    if val_recall < 0.85:
        print(f"  ⚠️  WARNING: Recall={val_recall:.1%} < 85%! Model may be missing fights.")

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping: no improvement for {PATIENCE} epochs.")
        break

print(f"\nDone. Best: ep {best_epoch}, f1={best_val_f1:.3f}, acc={best_val_acc:.1%}, recall={best_val_recall:.1%}")

## 10. Evaluation

In [ ]:
checkpoint = torch.load('best_finetuned.pt')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best from epoch {checkpoint['epoch']}")

val_loss, val_acc, val_f1, all_preds, all_labels, all_scores, all_logits, val_recall = eval_epoch(model, val_loader, criterion)

print(f"\nFinal Val Accuracy: {val_acc:.1%}")
print(f"Final Val F1: {val_f1:.3f}")
print(f"Final Val Recall: {val_recall:.1%}")

print("\n" + classification_report(
    all_labels, all_preds,
    labels=[0, 1],
    target_names=['Non-Fight', 'Fight'],
    digits=3,
    zero_division=0
))

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
print("Confusion Matrix:")
print(f"              Pred NF  Pred F")
print(f"  Actual NF   {cm[0][0]:>5}   {cm[0][1]:>5}")
print(f"  Actual F    {cm[1][0]:>5}   {cm[1][1]:>5}")

# Save for threshold optimization and temperature scaling
final_scores = np.array(all_scores)
final_labels = np.array(all_labels)
final_logits = np.array(all_logits)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
phase_split = EPOCHS_PHASE1

axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].axvline(phase_split - 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 1/2')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].axvline(phase_split - 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 1/2')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['val_f1'], label='Val F1', linewidth=2, color='green')
axes[2].axvline(phase_split - 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 1/2')
axes[2].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (ep {best_epoch})')
axes[2].set_title('Val F1'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('HPI-GCN-RP Fine-tuning Curves', fontsize=16)
plt.tight_layout()
plt.savefig('finetune_curves.png', dpi=150)
plt.show()

## 10.5 Threshold Optimization (Youden's J) + Temperature Scaling

Find the optimal classification threshold and calibrate confidence scores.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
from scipy.optimize import minimize_scalar

# ── Youden's J Statistic: find optimal threshold ──
print("=" * 50)
print("THRESHOLD OPTIMIZATION (Youden's J)")
print("=" * 50)

fpr, tpr, thresholds_roc = roc_curve(final_labels, final_scores)
j_scores = tpr - fpr
best_j_idx = np.argmax(j_scores)
optimal_threshold = float(thresholds_roc[best_j_idx])

print(f"\nYouden's J optimal threshold: {optimal_threshold:.4f}")
print(f"  At this threshold: TPR={tpr[best_j_idx]:.3f}, FPR={fpr[best_j_idx]:.3f}")
print(f"  Youden's J score: {j_scores[best_j_idx]:.3f}")

# Also find threshold for recall >= 95% (safety margin)
recall_95_idx = np.where(tpr >= 0.95)[0]
if len(recall_95_idx) > 0:
    threshold_95_recall = float(thresholds_roc[recall_95_idx[0]])
    print(f"\nThreshold for 95% recall: {threshold_95_recall:.4f}")
    print(f"  FPR at 95% recall: {fpr[recall_95_idx[0]]:.3f}")
else:
    threshold_95_recall = 0.3
    print(f"\nCould not find threshold for 95% recall, using {threshold_95_recall}")

# ── Temperature Scaling ──
print("\n" + "=" * 50)
print("TEMPERATURE SCALING")
print("=" * 50)

logits_tensor = torch.FloatTensor(final_logits)
labels_tensor = torch.FloatTensor(final_labels)

def nll_loss(T):
    """Negative log-likelihood with temperature T."""
    scaled = torch.sigmoid(logits_tensor / T)
    eps = 1e-7
    loss = -torch.mean(
        labels_tensor * torch.log(scaled + eps) +
        (1 - labels_tensor) * torch.log(1 - scaled + eps)
    )
    return loss.item()

result = minimize_scalar(nll_loss, bounds=(0.1, 10.0), method='bounded')
optimal_temperature = float(result.x)

print(f"Optimal temperature: {optimal_temperature:.4f}")
print(f"  NLL before (T=1.0): {nll_loss(1.0):.4f}")
print(f"  NLL after  (T={optimal_temperature:.2f}): {nll_loss(optimal_temperature):.4f}")

# ── Visualize ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve with threshold
axes[0].plot(fpr, tpr, linewidth=2, label='ROC')
axes[0].scatter([fpr[best_j_idx]], [tpr[best_j_idx]], c='red', s=100, zorder=5,
                label=f'Optimal (thr={optimal_threshold:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f"ROC Curve (AUC, J={j_scores[best_j_idx]:.3f})")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Calibrated vs uncalibrated scores
bins = np.linspace(0, 1, 20)
uncalibrated = torch.sigmoid(logits_tensor).numpy()
calibrated = torch.sigmoid(logits_tensor / optimal_temperature).numpy()
axes[1].hist(uncalibrated[final_labels == 0], bins=bins, alpha=0.5, label='Non-fight (T=1.0)', color='blue')
axes[1].hist(uncalibrated[final_labels == 1], bins=bins, alpha=0.5, label='Fight (T=1.0)', color='red')
axes[1].axvline(optimal_threshold, color='green', linestyle='--', linewidth=2, label=f'Threshold={optimal_threshold:.3f}')
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution with Optimal Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('threshold_calibration.png', dpi=150)
plt.show()

print(f"\n{'='*50}")
print(f"SUMMARY:")
print(f"  Optimal threshold: {optimal_threshold:.4f}")
print(f"  Temperature:       {optimal_temperature:.4f}")
print(f"  These will be saved in the exported checkpoint.")
print(f"{'='*50}")

## 11. Export Model

In [ ]:
export_path = "hpi_gcn_rp_finetuned.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'model_name': 'HPI-GCN-RP',
    'num_keypoints': NUM_KEYPOINTS,
    'num_people': NUM_PEOPLE,
    'in_channels': IN_CHANNELS,
    'sequence_len': SEQUENCE_LEN,
    'val_accuracy': best_val_acc,
    'val_f1': best_val_f1,
    'val_recall': best_val_recall,
    'epoch': best_epoch,
    'fine_tuned': True,
    'optimal_threshold': optimal_threshold,
    'temperature': optimal_temperature,
}, export_path)

file_size = os.path.getsize(export_path) / 1024
print(f"Exported: {export_path} ({file_size:.0f} KB)")
print(f"Val accuracy: {best_val_acc:.1%}")
print(f"Val F1: {best_val_f1:.3f}")
print(f"Val Recall: {best_val_recall:.1%}")
print(f"Optimal threshold: {optimal_threshold:.4f}")
print(f"Temperature: {optimal_temperature:.4f}")
print(f"\n{'='*50}")
print(f"DOWNLOAD THIS FILE")
print(f"Rename to: best_model.pth")
print(f"Put in: AiGuardian/models/best_model.pth")
print(f"{'='*50}")
print(f"\nThe model will automatically load threshold={optimal_threshold:.4f}")
print(f"and temperature={optimal_temperature:.4f} from the checkpoint.")